---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

### Konteks / Skenario

Manajemen "Kopi Nusantara" ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Tugasnya untuk membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

### Instruksi Pengerjaan
---
Pembuatan `SparkSession`

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder.appName("Tugas10").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")
print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

In [9]:
# (1) Mendefinisikan skema eksplisit dan membuat folder simulasi streaming baru 
folder = "tugas10_stream"
shutil.rmtree(folder, ignore_errors=True)
os.makedirs(folder, exist_ok=True)

skema = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])
print("Siap untuk latihan. Folder:", folder)
print()

# (2) Mendefinsikan readStream dari folder tersebut
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema) \
    .option("header", True) \
    .load(folder)
print("Streaming DataFrame berhasil didefinisikan.")
print("df_stream.isStreaming:", df_stream.isStreaming)

Siap untuk latihan. Folder: tugas10_stream

Streaming DataFrame berhasil didefinisikan.
df_stream.isStreaming: True


---
**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

In [10]:
# (1) Menambahkan kolom turunan total_harga
df_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# (2) Membuat streaming query 
agregasi = df_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_harga"), 
    count("pesanan_id").alias("jumlah_pesanan")
)

query = agregasi.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()
print("Transformasi berhasil didefinisikan.")
print("Streaming query dimulai! Status:", query.status)

Transformasi berhasil didefinisikan.
Streaming query dimulai! Status: {'message': 'Getting offsets from FileStreamSource[file:/home/thinkpad/praktikum-bigdata/tugas10_stream]', 'isDataAvailable': False, 'isTriggerActive': True}


---
**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

In [11]:
# (1) Mendefinisikan ulang readStream
df_stream2 = spark.readStream \
    .format("csv") \
    .schema(skema) \
    .option("header", True) \
    .load(folder)

# (2) Mengelompokkan total total_harga berdasarkan window waktu 2 menit
df_total2 = df_stream2.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi = df_total2.groupBy(
    window(col("waktu_pesan"), "2 minute")
).agg(
    spark_sum("total_harga").alias("total_harga")
)

# (3) Membuat streaming query kedua 
query2 = agregasi.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()
print("Streaming query window waktu dimulai.")

Streaming query window waktu dimulai.


---
**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [12]:
# Batch 1
pd.DataFrame({
    "pesanan_id": ["P1", "P2", "P3"],
    "nama_cabang": ["Magelang", "Solo", "Yogyakarta"],
    "nama_menu": ["Seblak", "Latte", "Americano"],
    "qty": [2, 5, 5],
    "harga_satuan": [20000, 22000, 15000],
    "waktu_pesan": ["2026-08-13 15:30:00", "2026-08-13 15:30:15", "2026-08-13 15:30:45"],
}).to_csv(f"{folder}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(6)  
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_harga DESC").show()

# Batch 2
pd.DataFrame({
    "pesanan_id": ["P4", "P5"],
    "nama_cabang": ["Yogyakarta", "Solo"],
    "nama_menu": ["Cappuccino", "Kopi Dolce"],
    "qty": [3, 2],
    "harga_satuan": [22000, 22000],
    "waktu_pesan": ["2026-08-14 15:30:00", "2026-08-14 15:30:15"],
}).to_csv(f"{folder}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)  
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_harga DESC").show()

# Batch 3
pd.DataFrame({
    "pesanan_id": ["P6", "P7"],
    "nama_cabang": ["Yogyakarta", "Magelang"],
    "nama_menu": ["Ayam Geprek", "Kopi Dolce"],
    "qty": [1, 5],
    "harga_satuan": [25000, 22000],
    "waktu_pesan": ["2026-08-15 15:30:15", "2026-08-15 15:30:45"],
}).to_csv(f"{folder}/batch_3.csv", index=False)

print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(6)  
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_harga DESC").show()

Batch 1 dikirim. Menunggu Spark memproses...


+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|       Solo|     110000|             1|
| Yogyakarta|      75000|             1|
|   Magelang|      40000|             1|
+-----------+-----------+--------------+

Batch 2 dikirim. Menunggu Spark memproses...
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|       Solo|     154000|             2|
| Yogyakarta|     141000|             2|
|   Magelang|      40000|             1|
+-----------+-----------+--------------+

Batch 3 dikirim. Menunggu Spark memproses...
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
| Yogyakarta|     166000|             3|
|       Solo|     154000|             2|
|   Magelang|     150000|             2|
+-----------+-----------+--------------+



---
**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

In [14]:
from datetime import datetime
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

# (1) Membuat streaming query ketiga
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
print("Streaming query ketiga (sink parquet) dimulai.")

# (2) Batch 4
waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ["P8", "P9"],
    "nama_cabang": ["Yogyakarta", "Magelang"],
    "nama_menu": ["Caramel Latte", "Latte"],
    "qty": [5, 6],
    "harga_satuan": [26000, 22000],
    "waktu_pesan": [waktu_sekarang] * 2,
}).to_csv(f"{folder}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu Spark memproses...")
time.sleep(10)  
print()

# (3) Membuktikan berkas Parquet dan membaca kembali sebagai DataFrame
!ls tugas10_output
print()
df = spark.read.parquet("tugas10_output")
df.orderBy("pesanan_id").show()
print("Jumlah baris:", df.count())
print("df_hasil.isStreaming:", df.isStreaming)

Streaming query ketiga (sink parquet) dimulai.
Batch 4 dikirim. Menunggu Spark memproses...

part-00000-3759636f-176b-41d1-adbe-5cad1bbe065a-c000.snappy.parquet
part-00001-69fef5b8-cbfa-4d57-a9d2-1211e5a03fcb-c000.snappy.parquet
part-00002-c0beeb8d-c561-4ac5-9c93-a0cea950df7f-c000.snappy.parquet
part-00003-fe2e3e1e-03c2-45ca-9642-6c1e9ae142cc-c000.snappy.parquet
_spark_metadata

+----------+-----------+-------------+---+------------+-------------------+
|pesanan_id|nama_cabang|    nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+-------------+---+------------+-------------------+
|        P1|   Magelang|       Seblak|  2|       20000|2026-08-13 15:30:00|
|        P2|       Solo|        Latte|  5|       22000|2026-08-13 15:30:15|
|        P3| Yogyakarta|    Americano|  5|       15000|2026-08-13 15:30:45|
|        P4| Yogyakarta|   Cappuccino|  3|       22000|2026-08-14 15:30:00|
|        P5|       Solo|   Kopi Dolce|  2|       22000|2026-08-14 15:30:15|
|        P

In [15]:
query.stop()
query2.stop()
query3.stop()
spark.stop()
print("Seluruh streaming query dihentikan, SparkSession ditutup.")

Seluruh streaming query dihentikan, SparkSession ditutup.
